In [1]:
# Synthetic oil-rate prediction
# All comments are in English, no special characters.

import numpy as np
import tensorflow as tf

# 1. Synthetic data -----------------------------------------------------------
np.random.seed(42)

# Features: reservoir pressure (psi), choke size (in), water cut (%)
n_samples = 1_000
p_res   = np.random.uniform(2_000, 4_500, size=n_samples)      # reservoir pressure
choke   = np.random.uniform(16, 64, size=n_samples)            # choke opening
w_cut   = np.random.uniform(0.0, 0.6, size=n_samples)          # water cut fraction

# Target: oil rate (bbl/d) — simple physical proxy plus noise
oil_rate = (
    0.25 * (p_res - 1_800)             # higher pressure → more flow
    + 8.0 * choke                      # larger choke → more flow
    - 500.0 * w_cut                    # water cut reduces oil
)
oil_rate = oil_rate + np.random.normal(0.0, 80.0, size=n_samples)  # measurement noise

X = np.stack([p_res, choke, w_cut], axis=1).astype("float32")
y = oil_rate.astype("float32")

# 2. Train / test split -------------------------------------------------------
split = int(0.8 * n_samples)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 3. Model definition ---------------------------------------------------------
model = tf.keras.Sequential([
    tf.keras.Input(shape =(3,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)                        # linear output for regression
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 4. Training -----------------------------------------------------------------
history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

# 5. Evaluation ---------------------------------------------------------------
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test MAE = {test_mae:0.1f} bbl/d")

# 6. Prediction for a new well condition --------------------------------------
new_sample = np.array([[3_200, 32, 0.25]], dtype="float32")   # [psi, in, fraction]
pred_rate  = model.predict(new_sample, verbose=0)
print(f"Predicted oil rate = {pred_rate[0,0]:0.1f} bbl/d")

# 7. Save model ---------------------------------------------------------------
model.save("oil_rate_model.keras")


Test MAE = 124.7 bbl/d
Predicted oil rate = 493.3 bbl/d
